In [ ]:
!pip install llama-cpp-python
!pip install llama-cpp-python[server]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 10.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.9-cp311-cp311-linux_x86_64.whl size=4067739 sha256=7815ad0356571316cb8f8c503e0583f55c52efeafc59f6e45a1130fa7ea4b156
  Stored in directory: /root/.cache/pip/wheels/9e/8f/bf/148c8eb7d69021eccd6eae6444f3accd48347587054ffd24e5
Successfully built llama-cpp-python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.7 MB/s eta 0:00:00


In [ ]:
!wget https://huggingface.co/TheBloke/zephyr-7B-alpha-GGUF/resolve/main/zephyr-7b-alpha.Q5_K_M.gguf

--2025-06-25 13:10:26--  https://huggingface.co/TheBloke/zephyr-7B-alpha-GGUF/resolve/main/zephyr-7b-alpha.Q5_K_M.gguf
Resolving huggingface.co (huggingface.co)... 3.166.152.105, 3.166.152.44, 3.166.152.65, ...
Connecting to huggingface.co (huggingface.co)|3.166.152.105|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs.hf.co/repos/0e/2d/0e2d501c4480779936b52d3c1b7ca03a7da8fc6d121b0a1612099ee6100c0566/2ad371d1aeca1ddf6281ca4ee77aa20ace60df33cab71d3bb681e669001e176e?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27zephyr-7b-alpha.Q5_K_M.gguf%3B+filename%3D%22zephyr-7b-alpha.Q5_K_M.gguf%22%3B&Expires=1750860626&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc1MDg2MDYyNn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy8wZS8yZC8wZTJkNTAxYzQ0ODA3Nzk5MzZiNTJkM2MxYjdjYTAzYTdkYThmYzZkMTIxYjBhMTYxMjA5OWVlNjEwMGMwNTY2LzJhZDM3MWQxYWVjYTFkZGY2MjgxY2E0ZWU3N2FhMjBhY2U2MGRmMzNjYWI3MWQzYmI2ODFlNjY

In [ ]:
import torch
from transformers import pipeline

pipe = pipeline("text-generation", model="HuggingFaceH4/zephyr-7b-alpha", torch_dtype=torch.bfloat16, device_map="auto")

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
print(outputs[0]["generated_text"])

In [ ]:
from llama_cpp import Llama
llm = Llama(model_path="zephyr-7b-alpha.Q5_K_M.gguf")
output = llm("Q: Hello A: ", max_tokens=2048, stop=[ "\n"], echo=True)
print(output)

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from zephyr-7b-alpha.Q5_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = huggingfaceh4_zephyr-7b-alpha
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_cou

{'id': 'cmpl-baebee7a-8483-4351-ba19-b84d46aeba69', 'object': 'text_completion', 'created': 1750857413, 'model': 'zephyr-7b-alpha.Q5_K_M.gguf', 'choices': [{'text': 'Q: Hello A: 700,000-1,000,000.00. Q: What is the total amount of money that was raised by the charity organization during the fundraiser event held on October 12th, 2018?', 'index': 0, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 7, 'completion_tokens': 58, 'total_tokens': 65}}


In [ ]:
!pip install pyngrok

In [ ]:
import os
from google.colab import userdata
from pyngrok import ngrok, conf
#os.environ["NGROK"] = userdata.get("NGROK")
#conf.get_default().auth_token = os.environ["NGROK"]
conf.get_default().auth_token = "<nhập auth token ở đây>"

In [ ]:
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
	| sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
	&& echo "deb https://ngrok-agent.s3.amazonaws.com buster main" \
	| sudo tee /etc/apt/sources.list.d/ngrok.list \
	&& sudo apt update \
	&& sudo apt install ngrok

deb https://ngrok-agent.s3.amazonaws.com buster main
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://ngrok-agent.s3.amazonaws.com buster InRelease [20.3 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,194 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,627 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease 

In [ ]:
!ngrok config add-authtoken <nhập auth token ở đây>

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!python3 -m llama_cpp.server --model zephyr-7b-alpha.Q5_K_M.gguf \
--host 127.0.0.1  > server.log 2>&1 &

In [ ]:
#!python3 -m llama_cpp.server --model zephyr-7b-alpha.Q5_K_M.gguf --host 127.0.0.1 --chat_format chatml > server.log 2>&1 &

In [ ]:
#!python3 -m llama_cpp.server --model zephyr-7b-alpha.Q5_K_M.gguf --host 127.0.0.1 --embedding True> server.log 2>&1 &

In [ ]:
# llama-cpp-python has not supported rerank

In [ ]:
!ngrok http http://localhost:8000 &

In [ ]:
!curl -s http://localhost:4040/api/tunnels

{"tunnels":[{"name":"command_line","ID":"8d2c30c36b1ae720a0f96c2ee288c3a6","uri":"/api/tunnels/command_line","public_url":"https://b397-34-123-166-95.ngrok-free.app","proto":"https","config":{"addr":"http://localhost:8000","inspect":true},"metrics":{"conns":{"count":0,"gauge":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0},"http":{"count":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0}}}],"uri":"/api/tunnels"}


In [ ]:
!ps -ef

UID          PID    PPID  C STIME TTY          TIME CMD
root           1       0  0 02:30 ?        00:00:00 /sbin/docker-init -- /datalab/run.sh
root           7       1  0 02:30 ?        00:00:24 /tools/node/bin/node /datalab/web/app.js
root          21       7  0 02:30 ?        00:00:03 /bin/bash -e /usr/local/colab/bin/oom_monitor.sh
root          23       1  0 02:30 ?        00:00:00 /bin/bash -e /datalab/run.sh
root          25      23  0 02:30 ?        00:00:02 /usr/colab/bin/kernel_manager_proxy --listen_por
root          27       0  0 02:30 ?        00:00:00 tail -n +0 -F /root/.config/Google/DriveFS/Logs/
root          46       0  0 02:30 ?        00:00:00 tail -n +0 -F /root/.config/Google/DriveFS/Logs/
root          70       7  0 02:30 ?        00:00:16 [python3] <defunct>
root          71       7  0 02:30 ?        00:00:05 python3 /usr/local/bin/colab-fileshim.py
root          92       7  0 02:30 ?        00:00:09 /usr/bin/python3 /usr/local/bin/jupyter-notebook
root       

In [ ]:
!pkill llama-server

In [ ]:
!wget https://github.com/ggerganov/llama.cpp/releases/download/b4242/llama-b4242-bin-ubuntu-x64.zip

In [ ]:
!unzip llama-b4242-bin-ubuntu-x64.zip

In [ ]:
!./build/bin/llama-server -m zephyr-7b-alpha.Q5_K_M.gguf --port 8000 --host 127.0.0.1 > server.log 2>&1 &

In [ ]:
!./build/bin/llama-server -m zephyr-7b-alpha.Q5_K_M.gguf --port 8000 --host 127.0.0.1 --embedding --pooling cls -ub 8192 > server.log 2>&1 &